## はじめに

このノートブックでは、Cortex Agent、Snowflake Managed MCP Server、およびProgrammatic Access Token（PAT）を作成します。

**主な処理内容:**
1. Semantic Viewの作成（Cortex Analyst用）
2. Cortex Agentの作成（Cortex Analyst + Cortex Searchツール）
3. Snowflake Managed MCP Serverの作成
4. Programmatic Access Token（PAT）の発行

---

### 全体アーキテクチャ

```
MCPクライアント（Claude Code、ChatGPT Enterprise、Cursor等）
        │
        ▼ MCP Protocol（OAuth/PAT認証）
┌───────────────────────────────────────────────┐
│     Snowflake Managed MCP Server              │
│                     │                         │
│     ┌───────────────┴───────────────┐         │
│     ▼                               ▼         │
│  Cortex Agent                  (他のツール)    │
│     │                                         │
│  ┌──┴──────────────────────┐                  │
│  ▼                         ▼                  │
│ Cortex Analyst         Cortex Search          │
│ (Semantic View)        (4 Services)           │
│     │                      │                  │
│     ▼                      ▼                  │
│ 構造化データ            非構造化データ          │
│ (売上・顧客等)          (FAQ・SNS等)           │
└───────────────────────────────────────────────┘
```

**MCP（Model Context Protocol）とは:**
- AIエージェントがビジネスアプリケーションや外部データシステムと安全に連携するためのオープンソース標準
- Snowflake Managed MCP Serverを使うことで、インフラ構築なしでAIエージェントがSnowflakeのデータにアクセス可能

In [ ]:
-- ============================================================================
-- 環境設定
-- ============================================================================
USE WAREHOUSE MCP_HANDSON_WH;
USE SCHEMA MCP_HANDSON_DB.ANALYTICS_SCHEMA;

## 1. 前提条件の確認

Part 1で作成したCortex Search Serviceが存在することを確認します。

In [ ]:
-- ============================================================================
-- Cortex Search Serviceの確認
-- ============================================================================
SHOW CORTEX SEARCH SERVICES IN SCHEMA MCP_HANDSON_DB.ANALYTICS_SCHEMA;

## 2. Semantic Viewの作成

Cortex Analyst向けのセマンティックビューを作成します。

**Semantic Viewとは:**
- ビジネスメトリクスやエンティティの関係を定義
- 自然言語クエリ（Text-to-SQL）を可能にする
- Cortex Agentのツールとして利用可能

**対象テーブル:**
- dim_customers: 顧客マスタ
- dim_products: 商品マスタ
- fact_orders: 注文トランザクション
- fact_payments: 決済トランザクション
- fact_web_logs: Webアクセスログ

In [ ]:
-- ============================================================================
-- Semantic View の作成
-- ============================================================================
CREATE OR REPLACE SEMANTIC VIEW MCP_HANDSON_DB.ANALYTICS_SCHEMA.EC_ANALYSIS_SEMANTIC_VIEW
    COMMENT = 'ECサイトの売上・顧客・商品・Webログを分析するためのセマンティックビュー'
    TABLES (
        customers AS MCP_HANDSON_DB.ANALYTICS_SCHEMA.dim_customers 
            PRIMARY KEY (customer_id),
        products AS MCP_HANDSON_DB.ANALYTICS_SCHEMA.dim_products 
            PRIMARY KEY (product_id),
        orders AS MCP_HANDSON_DB.ANALYTICS_SCHEMA.fact_orders 
            PRIMARY KEY (order_id),
        payments AS MCP_HANDSON_DB.ANALYTICS_SCHEMA.fact_payments 
            PRIMARY KEY (payment_id),
        web_logs AS MCP_HANDSON_DB.ANALYTICS_SCHEMA.fact_web_logs 
            PRIMARY KEY (log_id)
    )
    RELATIONSHIPS (
        orders_to_customers AS orders (customer_id) REFERENCES customers,
        orders_to_products AS orders (product_id) REFERENCES products,
        payments_to_orders AS payments (order_id) REFERENCES orders
    )
    FACTS (
        orders.order_total AS total_amount,
        payments.pay_amt AS payment_amount
    )
    DIMENSIONS (
        customers.cust_name AS CONCAT(last_name, first_name),
        products.prod_name AS product_name,
        orders.order_date AS order_datetime
    );

In [ ]:
-- ============================================================================
-- Semantic Viewの確認
-- ============================================================================
SHOW SEMANTIC VIEWS IN SCHEMA MCP_HANDSON_DB.ANALYTICS_SCHEMA;

## 3. Cortex Agentの作成

Cortex AnalystとCortex Searchをツールとして持つエージェントを作成します。

**エージェント設定:**
- オブジェクト名: `MCP_ANALYTICS_AGENT`
- 表示名: `MCP分析エージェント`
- モデル: `auto`（最新のモデルが自動選択）

**含まれるツール:**

| ツール名 | 種別 | 対象 |
|---------|------|------|
| EC_Sales_Analysis | Semantic View | 売上・顧客・商品データ |
| FAQ_Search | Cortex Search | FAQドキュメント |
| Operation_Manual_Search | Cortex Search | 業務マニュアル |
| Voice_Log_Search | Cortex Search | 音声ログ要約 |
| SNS_Mention_Search | Cortex Search | SNS投稿 |

In [ ]:
-- ============================================================================
-- Cortex Agent の作成（全ツール含む）
-- ============================================================================
CREATE OR REPLACE AGENT MCP_ANALYTICS_AGENT
  COMMENT = 'ECサイトの売上・顧客・VoC分析を自然言語で行うエージェントです。'
  PROFILE = '{"display_name": "MCP分析エージェント", "color": "blue"}'
FROM SPECIFICATION $$
models:
  orchestration: auto

instructions:
  orchestration: |
    あなたはECサイトの分析アシスタントです。
    ユーザーの質問に対して、以下の手順で適切なツールを選択してください。
    
    1. 質問の種類を判断する
       - 売上・注文・顧客・商品に関する数値分析 → EC_Sales_Analysis（Semantic View）を使用
       - 返品・配送・支払いなどのFAQ → FAQ_Search を使用
       - 業務手順・対応方法 → Operation_Manual_Search を使用
       - 過去の問い合わせ事例 → Voice_Log_Search を使用
       - SNSの評判・口コミ → SNS_Mention_Search を使用
    
    2. 複合的な質問の場合は、複数のツールを順番に使用する
    
    3. 検索結果が不十分な場合は、別のツールを試すか、ユーザーに追加情報を求める
    
    4. データの期間に注意する
       - 売上・注文データは2024年のデータです
  
  response: |
    以下のルールに従って応答してください。
    
    【口調・スタイル】
    - 丁寧語（です・ます調）で回答する
    - 専門用語は必要に応じて簡単な説明を添える
    - 回答は簡潔にまとめつつ、必要な情報は漏らさない
    
    【数値・データの表示】
    - 金額は3桁区切りで表示（例：1,234,567円）
    - パーセンテージは小数点第1位まで表示（例：12.3%）
    - 日付は YYYY年MM月DD日 形式で表示
    
    【回答の構成】
    - まず結論や要点を述べる
    - 必要に応じて詳細データや根拠を示す
    - 追加で確認できることがあれば提案する
    
    【注意事項】
    - 検索結果がない場合は、その旨を明確に伝える
    - 推測や不確実な情報には「〜と考えられます」を使用

tools:
  # Semantic View（売上・顧客分析）
  - tool_spec:
      type: cortex_analyst_text_to_sql
      name: EC_Sales_Analysis
      description: |
        ECサイトの売上、注文、顧客、商品、決済データを分析します。
        売上推移、カテゴリ別売上、顧客セグメント分析、購買傾向などの質問に回答できます。
        データは2024年のものです。

  # Cortex Search（FAQドキュメント）
  - tool_spec:
      type: cortex_search
      name: FAQ_Search
      description: |
        ECサイトのよくある質問（FAQ）から回答を検索します。
        返品・交換、配送、支払い、会員登録などに関する質問に対応します。

  # Cortex Search（業務マニュアル）
  - tool_spec:
      type: cortex_search
      name: Operation_Manual_Search
      description: |
        カスタマーサポート業務の運営マニュアルから手順や対応方法を検索します。
        クレーム対応、返品処理、エスカレーション手順などの業務フローを参照できます。

  # Cortex Search（音声ログ）
  - tool_spec:
      type: cortex_search
      name: Voice_Log_Search
      description: |
        コールセンターの過去の通話履歴（要約）から類似事例を検索します。
        過去の問い合わせ対応事例やクレーム対応履歴を参照できます。

  # Cortex Search（SNS投稿）
  - tool_spec:
      type: cortex_search
      name: SNS_Mention_Search
      description: |
        SNS（Twitter/Instagram）上の関連投稿から顧客の声を検索します。
        商品の評判、ブランドイメージ、改善要望などのVoC情報を参照できます。

tool_resources:
  # Semantic Viewの設定
  EC_Sales_Analysis:
    semantic_view: MCP_HANDSON_DB.ANALYTICS_SCHEMA.EC_ANALYSIS_SEMANTIC_VIEW

  # FAQ検索の設定
  FAQ_Search:
    search_service: MCP_HANDSON_DB.ANALYTICS_SCHEMA.SEARCH_FAQ
    max_results: 5

  # 業務マニュアル検索の設定
  Operation_Manual_Search:
    search_service: MCP_HANDSON_DB.ANALYTICS_SCHEMA.SEARCH_OPERATION_MANUALS
    max_results: 5

  # 音声ログ検索の設定
  Voice_Log_Search:
    search_service: MCP_HANDSON_DB.ANALYTICS_SCHEMA.SEARCH_VOICE_LOGS
    max_results: 5

  # SNS投稿検索の設定
  SNS_Mention_Search:
    search_service: MCP_HANDSON_DB.ANALYTICS_SCHEMA.SEARCH_SNS_MENTIONS
    max_results: 10
$$;

In [ ]:
-- ============================================================================
-- 作成したエージェントの確認
-- ============================================================================
SHOW AGENTS IN SCHEMA MCP_HANDSON_DB.ANALYTICS_SCHEMA;

In [ ]:
-- ============================================================================
-- エージェントの詳細確認
-- ============================================================================
DESCRIBE AGENT MCP_ANALYTICS_AGENT;

## 4. Snowflake Managed MCP Serverの作成

Cortex AgentをMCPクライアントから呼び出せるようにするMCP Serverを作成します。

**MCP Serverとは:**
- Model Context Protocol (MCP) に準拠したサーバー
- AIエージェント（Claude Desktop、Cursor等）がSnowflakeのデータに安全にアクセスするためのインターフェース
- OAuth認証またはPAT認証に対応

**サポートされるツールタイプ:**
- `CORTEX_AGENT_RUN`: Cortex Agentをツールとして呼び出し
- `CORTEX_ANALYST_MESSAGE`: Cortex Analystを直接呼び出し
- `CORTEX_SEARCH_SERVICE_QUERY`: Cortex Searchを直接呼び出し
- `SYSTEM_EXECUTE_SQL`: SQLクエリを実行
- `GENERIC`: UDFやストアドプロシージャを呼び出し

In [ ]:
-- ============================================================================
-- Snowflake Managed MCP Server の作成
-- Cortex Agentをツールとして公開
-- ============================================================================
CREATE OR REPLACE MCP SERVER MCP_ANALYTICS_SERVER
FROM SPECIFICATION $$
tools:
  - title: "EC分析エージェント"
    name: "ec-analytics-agent"
    type: "CORTEX_AGENT_RUN"
    identifier: "MCP_HANDSON_DB.ANALYTICS_SCHEMA.MCP_ANALYTICS_AGENT"
    description: |
      ECサイトのデータを分析するAIエージェントです。
      売上・顧客・商品の構造化データと、FAQ・マニュアル・SNSの非構造化データを
      横断的に分析できます。
      
      質問例:
      - 「2024年12月の売上上位10商品を教えて」
      - 「返品ポリシーについて教えて」
      - 「配送遅延に関する過去の問い合わせを検索して」
      - 「SNSでの商品の評判を教えて」
$$;

In [ ]:
-- ============================================================================
-- MCP Serverの確認
-- ============================================================================
SHOW MCP SERVERS IN SCHEMA MCP_HANDSON_DB.ANALYTICS_SCHEMA;

In [ ]:
-- ============================================================================
-- MCP Serverの詳細確認
-- ============================================================================
DESCRIBE MCP SERVER MCP_ANALYTICS_SERVER;

## 5. Programmatic Access Token（PAT）の発行

MCPクライアントからSnowflakeに接続するためのPATを発行します。

**PATとは:**
- Programmatic Access Token（プログラマティックアクセストークン）
- パスワードやSSO認証の代わりに使用できる認証トークン
- REST API、ドライバー、MCPクライアントなどから認証に使用

**セキュリティ上の注意:**
- PATは一度しか表示されないため、安全に保管してください
- 最小権限の原則に従い、必要なロールのみを指定してください
- 適切な有効期限を設定してください

In [ ]:
-- ============================================================================
-- Programmatic Access Token（PAT）の発行
-- 注意: トークンは一度しか表示されないため、安全に保管してください
-- ============================================================================
ALTER USER ADD PROGRAMMATIC ACCESS TOKEN MCP_HANDSON_PAT
  DAYS_TO_EXPIRY = 30
  COMMENT = 'MCP Handson用のPAT - MCPクライアント接続用';

### ⚠️ 重要: PATの保存

上記のコマンドを実行すると、以下のような出力が表示されます：

```
+---------------------------------------+
| PAT                                   |
+---------------------------------------+
| pt_xxxxxxxx-xxxx-xxxx-xxxx-xxxxxxxxxx |
+---------------------------------------+
```

**このトークンは一度しか表示されません。**  
必ず安全な場所に保存してください。

## 6. MCPクライアントからの接続

作成したMCP ServerにMCPクライアントから接続する方法を説明します。

---

### 6-1. エンドポイントURLの確認

MCP Serverのエンドポイントは以下の形式です：

```
https://<account_url>/api/v2/databases/MCP_HANDSON_DB/schemas/ANALYTICS_SCHEMA/mcp-servers/MCP_ANALYTICS_SERVER
```

**<account_url>の形式:**
- `<orgname>-<account_name>.snowflakecomputing.com`

例: `myorg-myaccount.snowflakecomputing.com`

In [ ]:
-- ============================================================================
-- アカウントURL情報の確認
-- ============================================================================
SELECT CURRENT_ORGANIZATION_NAME() AS org_name,
       CURRENT_ACCOUNT_NAME() AS account_name,
       CURRENT_ORGANIZATION_NAME() || '-' || CURRENT_ACCOUNT_NAME() || '.snowflakecomputing.com' AS account_url;

### 6-2. Cursor での設定

Cursorの設定ファイル（`~/.cursor/mcp.json` または プロジェクト内の `.cursor/mcp.json`）に以下を追加します：

```json
{
  "mcpServers": {
    "snowflake-mcp": {
      "url": "https://<account_url>/api/v2/databases/MCP_HANDSON_DB/schemas/ANALYTICS_SCHEMA/mcp-servers/MCP_ANALYTICS_SERVER",
      "headers": {
        "Authorization": "Bearer <your_pat_token>"
      }
    }
  }
}
```

**設定項目:**
- `<account_url>`: 上記で確認したアカウントURL
- `<your_pat_token>`: Step 5で発行したPATトークン

**設定後の手順:**
1. Cursorを再起動
2. チャットで `@snowflake-mcp` と入力してMCPサーバーが認識されていることを確認

### 6-3. Claude Code での設定

Claude Codeでは、以下のコマンドでMCPサーバーを追加します：

```bash
claude mcp add snowflake-mcp \
  --transport http \
  --url "https://<account_url>/api/v2/databases/MCP_HANDSON_DB/schemas/ANALYTICS_SCHEMA/mcp-servers/MCP_ANALYTICS_SERVER" \
  --header "Authorization: Bearer <your_pat_token>"
```

**設定項目:**
- `<account_url>`: 上記で確認したアカウントURL
- `<your_pat_token>`: Step 5で発行したPATトークン

**設定確認:**
```bash
# 登録済みMCPサーバーの一覧を確認
claude mcp list
```

**使用例:**
```bash
# Claude Codeを起動して質問
claude
> 2024年12月の売上上位10商品を教えて
```

### 6-4. ChatGPT Enterprise での設定

ChatGPT EnterpriseでMCPサーバーを利用するには、管理者がMCP接続を設定する必要があります。

**管理者による設定手順:**

1. ChatGPT Enterprise の管理コンソールにログイン
2. **Settings** → **Integrations** → **MCP Servers** を開く
3. **Add MCP Server** をクリック
4. 以下の情報を入力：

| 設定項目 | 値 |
|---------|----|
| Name | `Snowflake Analytics` |
| URL | `https://<account_url>/api/v2/databases/MCP_HANDSON_DB/schemas/ANALYTICS_SCHEMA/mcp-servers/MCP_ANALYTICS_SERVER` |
| Authentication | `Bearer Token` |
| Token | `<your_pat_token>` |

5. **Test Connection** で接続を確認
6. **Save** で保存

**ユーザー側の使用方法:**

1. ChatGPTの新規チャットを開く
2. メッセージ入力欄の左側にある **+** アイコンをクリック
3. **MCP Servers** から `Snowflake Analytics` を選択
4. 質問を入力（例：「2024年12月の売上上位10商品を教えて」）

> **注意**: ChatGPT EnterpriseのMCP機能は組織の設定により利用可能かどうかが異なります。

### 6-5. 接続テスト用の質問例

MCPクライアントから以下のような質問を試してみてください：

**売上分析（Semantic View経由）:**
- 「2024年12月の売上上位10商品を教えて」
- 「カテゴリ別の売上構成比を教えて」
- 「新規顧客とリピーターの購買金額の違いは？」

**FAQ検索（Cortex Search経由）:**
- 「返品ポリシーについて教えて」
- 「配送料について教えて」

**マニュアル検索（Cortex Search経由）:**
- 「クレーム対応の手順を教えて」
- 「返品処理の方法は？」

**音声ログ検索（Cortex Search経由）:**
- 「配送遅延に関する過去の問い合わせを検索して」
- 「ネガティブな問い合わせの事例を教えて」

**SNS分析（Cortex Search経由）:**
- 「SNSでの商品の評判を教えて」
- 「Twitterでのポジティブな投稿を検索して」

## 7. クリーンアップ（オプション）

ハンズオン終了後、作成したリソースを削除する場合は以下を実行してください。

In [ ]:
-- ============================================================================
-- クリーンアップ: PATの削除
-- ============================================================================
-- ALTER USER DROP PROGRAMMATIC ACCESS TOKEN MCP_HANDSON_PAT;

In [ ]:
-- ============================================================================
-- クリーンアップ: 作成したオブジェクトの削除
-- ============================================================================
-- DROP MCP SERVER MCP_ANALYTICS_SERVER;
-- DROP AGENT MCP_ANALYTICS_AGENT;
-- DROP SEMANTIC VIEW EC_ANALYSIS_SEMANTIC_VIEW;
-- DROP CORTEX SEARCH SERVICE SEARCH_FAQ;
-- DROP CORTEX SEARCH SERVICE SEARCH_OPERATION_MANUALS;
-- DROP CORTEX SEARCH SERVICE SEARCH_VOICE_LOGS;
-- DROP CORTEX SEARCH SERVICE SEARCH_SNS_MENTIONS;
-- DROP DATABASE MCP_HANDSON_DB;

## まとめ

このノートブックでは、Snowflake Managed MCP Serverを構築しました。

### 作成したオブジェクト

| オブジェクト | 名前 | 説明 |
|------------|------|------|
| Semantic View | EC_ANALYSIS_SEMANTIC_VIEW | 売上・顧客・商品データの分析用 |
| Cortex Agent | MCP_ANALYTICS_AGENT | Semantic View + Cortex Search を統合 |
| MCP Server | MCP_ANALYTICS_SERVER | MCPクライアントからのアクセスポイント |
| PAT | MCP_HANDSON_PAT | 認証用トークン |

### アーキテクチャのポイント

- **MCP Server**: MCPプロトコルに準拠したエンドポイントを提供
- **Cortex Agent**: 構造化データ（Semantic View）と非構造化データ（Cortex Search）を統合
- **PAT認証**: セキュアな認証方式でMCPクライアントからアクセス

### 次のステップ

- MCPクライアント（Cursor、Claude Desktop等）から接続して分析を実行
- 追加のツール（UDF、ストアドプロシージャ等）をMCP Serverに登録
- OAuth認証の設定（本番環境向け）